<a href="https://colab.research.google.com/github/nickrafael9-maker/clase01/blob/main/proyecto_ia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install pgmpy -q

print("[OK] Librería pgmpy instalada correctamente en el entorno de Colab.")

[OK] Librería pgmpy instalada correctamente en el entorno de Colab.


In [6]:
import warnings
warnings.filterwarnings('ignore')

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination


modelo_minibus = DiscreteBayesianNetwork([
    ('Clima_Frio', 'Bateria_Baja'),
    ('Desgaste_Mecanico', 'Falla_Arranque'),
    ('Desgaste_Mecanico', 'Falla_Alternador'),
    ('Falla_Alternador', 'Bateria_Baja'),
    ('Bateria_Baja', 'No_Arranca'),
    ('Falla_Arranque', 'No_Arranca')
])

print("[OK] Topología de la Red Bayesiana construida (DAG).")

[OK] Topología de la Red Bayesiana construida (DAG).


In [7]:

cpd_clima = TabularCPD(variable='Clima_Frio', variable_card=2,
                       values=[[0.30], [0.70]])

cpd_desgaste = TabularCPD(variable='Desgaste_Mecanico', variable_card=2,
                          values=[[0.60], [0.40]])

cpd_alternador = TabularCPD(
    variable='Falla_Alternador', variable_card=2,
    values=[[0.90, 0.30],
            [0.10, 0.70]],
    evidence=['Desgaste_Mecanico'], evidence_card=[2])

cpd_arranque = TabularCPD(
    variable='Falla_Arranque', variable_card=2,
    values=[[0.95, 0.40],
            [0.05, 0.60]],
    evidence=['Desgaste_Mecanico'], evidence_card=[2])

cpd_bateria = TabularCPD(
    variable='Bateria_Baja', variable_card=2,
    values=[[0.90, 0.40, 0.30, 0.10],
            [0.10, 0.60, 0.70, 0.90]],
    evidence=['Clima_Frio', 'Falla_Alternador'], evidence_card=[2, 2])

cpd_no_arranca = TabularCPD(
    variable='No_Arranca', variable_card=2,
    values=[[0.99, 0.10, 0.15, 0.01],
            [0.01, 0.90, 0.85, 0.99]],
    evidence=['Bateria_Baja', 'Falla_Arranque'], evidence_card=[2, 2])

modelo_minibus.add_cpds(cpd_clima, cpd_desgaste, cpd_alternador,
                        cpd_arranque, cpd_bateria, cpd_no_arranca)

if modelo_minibus.check_model():
    print("[OK] Tablas CPT cargadas. Modelo matemáticamente validado y listo para inferencias.")
else:
    print("[ERROR] Existe una inconsistencia matemática en las tablas CPT.")

[OK] Tablas CPT cargadas. Modelo matemáticamente validado y listo para inferencias.


In [8]:
inferencia = VariableElimination(modelo_minibus)

def imprimir_resultados(titulo, resultado):
    print("=" * 60)
    print(f" CASO: {titulo}")
    print("-" * 60)
    print(resultado)
    print("=" * 60, "\n")

res_diagnostico_bat = inferencia.query(
    variables=['Bateria_Baja'],
    evidence={'No_Arranca': 1}
)
imprimir_resultados("Diagnóstico - Probabilidad de Batería Baja al No Arrancar", res_diagnostico_bat)

res_diagnostico_alt = inferencia.query(
    variables=['Falla_Alternador'],
    evidence={'No_Arranca': 1}
)
imprimir_resultados("Diagnóstico - Probabilidad de Falla de Alternador", res_diagnostico_alt)

res_predictivo = inferencia.query(
    variables=['No_Arranca'],
    evidence={'Clima_Frio': 1, 'Desgaste_Mecanico': 1}
)
imprimir_resultados("Predicción - Probabilidad de Falla general (Frío + Desgaste)", res_predictivo)

res_aislado = inferencia.query(
    variables=['No_Arranca'],
    evidence={'Falla_Arranque': 1}
)
imprimir_resultados("Evaluación de Impacto - Arranque Dañado", res_aislado)

 CASO: Diagnóstico - Probabilidad de Batería Baja al No Arrancar
------------------------------------------------------------
+-----------------+---------------------+
| Bateria_Baja    |   phi(Bateria_Baja) |
+=================+=====================+
| Bateria_Baja(0) |              0.1196 |
+-----------------+---------------------+
| Bateria_Baja(1) |              0.8804 |
+-----------------+---------------------+

 CASO: Diagnóstico - Probabilidad de Falla de Alternador
------------------------------------------------------------
+---------------------+-------------------------+
| Falla_Alternador    |   phi(Falla_Alternador) |
+=====================+=========================+
| Falla_Alternador(0) |                  0.5489 |
+---------------------+-------------------------+
| Falla_Alternador(1) |                  0.4511 |
+---------------------+-------------------------+

 CASO: Predicción - Probabilidad de Falla general (Frío + Desgaste)
------------------------------------------